# TinyBanglaBERT – final model evaluation (R3: pretrained init + distillation)
GPU on, Internet on. Inputs to attach: (1) dataset with the newest `tiny_bert-fixed.zip`, (2) the **output of your ablation notebook run** (it contains the teacher and all `R*_s42` student folders).
Runs: McNemar tests, efficiency benchmark (latency/size/ONNX/INT8), cross-domain evaluation, SHAP/LIME. Each step is independent: if one fails, the rest still run.

In [ ]:
import glob, os, shutil, json, traceback
cands=[h for h in glob.glob('/kaggle/input/**/tiny_bert-main', recursive=True)
       if os.path.exists(os.path.join(h,'scripts','train.py')) and 'small_random' in open(os.path.join(h,'scripts','train.py')).read()]
assert cands, "No tiny_bert-main with the NEW code found: upload newest tiny_bert-fixed.zip as a dataset version and add it."
print("Using code from:", cands[0])
shutil.rmtree('/kaggle/working/tiny_bert-main', ignore_errors=True); shutil.copytree(cands[0], '/kaggle/working/tiny_bert-main')
os.chdir('/kaggle/working/tiny_bert-main'); os.makedirs('experiments', exist_ok=True); os.makedirs('data/processed', exist_ok=True)

In [ ]:
!pip install -q omegaconf lime shap onnx onnxruntime onnxscript optuna statsmodels tabulate psutil

## Restore teacher, splits and the seed-42 student runs from the previous output

In [ ]:
def restore(name):
    hit = glob.glob(f'/kaggle/input/**/experiments/{name}/model.safetensors', recursive=True)
    assert hit, f"{name} not found in inputs - attach the ablation notebook output"
    shutil.copytree(os.path.dirname(hit[0]), f'experiments/{name}', dirs_exist_ok=True); print('restored', name)
for n in ['teacher','R0_random_noKD_s42','R1_pretrained_noKD_s42','R2_random_KD_s42','R3_pretrained_KD_s42']: restore(n)
sp = glob.glob('/kaggle/input/**/data/processed/splits.json', recursive=True); assert sp
shutil.copy(sp[0],'data/processed/splits.json')
shutil.copytree('experiments/R3_pretrained_KD_s42','experiments/tinybanglabert_final', dirs_exist_ok=True)   # final model = R3, seed 42
m=json.load(open('data/processed/splits.json'))['metadata']; print({k:m[k] for k in ('train_size','val_size','test_size')})
from transformers import AutoModelForSequenceClassification as M
fm=M.from_pretrained('experiments/tinybanglabert_final'); print('FINAL MODEL params (M):', round(sum(p.numel() for p in fm.parameters())/1e6,2))
print('FINAL test metrics:', {k:v for k,v in json.load(open('experiments/tinybanglabert_final/test_metrics.json')).items() if k in ('accuracy','f1_macro','f1_clickbait')})

## 1. McNemar tests on the test set (paired, exact)

In [ ]:
try:
    from src.utils import load_config
    from src.data.load import load_raw
    from src.data.clean import clean_dataframe
    from src.data.split import apply_splits, load_splits
    from src.evaluation.mcnemar_test import get_model_test_predictions, run_pairwise_mcnemar_test
    cfg=load_config('configs/base.yaml')
    df,_=load_raw(cfg, hf_dataset_id=cfg.data.hf_dataset_id); df=clean_dataframe(df, headline_col=cfg.data.text_col)
    _,_,test_df=apply_splits(df, load_splits(cfg)); H=test_df['headline'].tolist(); y=test_df['label'].tolist()
    paths={'Teacher':'experiments/teacher','R3 pretrained+KD (final)':'experiments/R3_pretrained_KD_s42','R1 pretrained, no KD':'experiments/R1_pretrained_noKD_s42',
           'R2 random+KD':'experiments/R2_random_KD_s42','R0 random, no KD':'experiments/R0_random_noKD_s42'}
    P={k:get_model_test_predictions(v,H) for k,v in paths.items()}
    pairs=[('Teacher','R3 pretrained+KD (final)'),('R3 pretrained+KD (final)','R1 pretrained, no KD'),('R2 random+KD','R0 random, no KD'),
           ('R3 pretrained+KD (final)','R2 random+KD'),('R1 pretrained, no KD','R0 random, no KD')]
    import pandas as pd; rows=[]
    for a,b in pairs:
        r=run_pairwise_mcnemar_test(y,P[a],P[b],a,b,exact=True); c=r['counts']
        rows.append({'A':a,'B':b,'A only correct':c['a_only_correct_n01'],'B only correct':c['b_only_correct_n10'],'p-value':r['p_value_formatted'],'significant (0.05)':r['is_significant_alpha_05']})
    t=pd.DataFrame(rows); print(t.to_string(index=False)); t.to_csv('experiments/mcnemar_final.csv',index=False)
except Exception: traceback.print_exc()

## 2. Efficiency benchmark (size, CPU/GPU latency, ONNX, INT8)\nNote: numbers depend on the Kaggle hardware - state the hardware in the thesis.

In [ ]:
!python -m scripts.benchmark --n-iterations 200

In [ ]:
print(open('experiments/efficiency_report.md').read()[:4000])

## 3. Zero-shot cross-domain evaluation (BaitBuster-Bangla, 10,000 headlines)

In [ ]:
try:
    from src.evaluation.crossdomain_eval import run_crossdomain_evaluation
    df_cd, rpt = run_crossdomain_evaluation('abdalimran/BaitBuster-Bangla'); print(df_cd.to_string(index=False))
except Exception: traceback.print_exc()

## 4. Explainability (SHAP + LIME)

In [ ]:
!python -m scripts.explain --generate-thesis-samples --sample-size 15

In [ ]:
!cd /kaggle/working/tiny_bert-main && zip -qr /kaggle/working/final_eval.zip experiments -x "*/checkpoints/*" -x "experiments/teacher/*.safetensors" -x "experiments/R*/*.safetensors" -x "experiments/tinybanglabert_final/*.safetensors"
!ls -lh /kaggle/working/final_eval.zip